Egocentric Real-Time Indian Sign Language Recognition on Ray-Ban Meta Smart Glasses

droid cam setup and stream phone cam in system
rtm pose or media pipe feature extraction both option to be available


env: meta_isl
code: meta_isl\Scripts\activate

In [1]:
# !pip install mediapipe opencv-python numpy matplotlib

In [2]:
# !pip install -U openmim

# !mim install mmengine
# !mim install "mmcv>=2.0.0"
# !mim install mmdet
# !mim install mmpose

In [3]:
# # Remove existing installation
# !pip uninstall -y mediapipe

# # Install stable version
# !pip install mediapipe==0.10.14

In [4]:
# !pip install einops
# Run once, then comment out


In [5]:
import cv2
import requests
import numpy as np
import mediapipe as mp

from PIL import Image
from IPython.display import display, clear_output

In [6]:
#For Driod Cam
#URL = "http://192.168.0.5:4747/video"

In [7]:
#For Driod Cam Connection
def connect_droidcam(url):

    print("Connecting to DroidCam...")

    stream = requests.get(
        url,
        stream=True
    )

    if stream.status_code != 200:
        raise Exception(
            f"Connection failed: {stream.status_code}"
        )

    print("Connected!")

    return stream

In [8]:
#To check for Driod Connection
#stream = connect_droidcam(URL)
#buffer = b""

In [9]:
#for meta glasses dependency
# !pip install websockets nest_asyncio

In [10]:
#for imports for meta glasses stream
import asyncio
import threading
import queue
import websockets
import nest_asyncio

# Patch Jupyter's running event loop so asyncio.run() works inside notebook
nest_asyncio.apply()


In [11]:
#To check or torch import
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, json, os
from collections import deque
from einops import rearrange, repeat

device = torch.device('cpu')
print(f"PyTorch: {torch.__version__} | Device: {device}")


PyTorch: 2.12.1+cpu | Device: cpu


In [17]:
#To check for mediapipe holistics

import mediapipe as mp
print(mp.__version__)          # should be 0.10.14
print(hasattr(mp, 'solutions')) # should be True


0.10.14
True


In [13]:
mp_holistic = mp.solutions.holistic

holistic = mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

In [14]:
def extract_pose(frame_rgb):

    results = holistic.process(frame_rgb)

    return {
        "pose": results.pose_landmarks,
        "left_hand": results.left_hand_landmarks,
        "right_hand": results.right_hand_landmarks,
        "face": results.face_landmarks
    }

In [15]:
#returns a flat array with all the coordinates concatenated: not suitable for HDGCN
def mediapipe_to_vector(results):

    features = []

    # Pose
    if results["pose"]:

        for lm in results["pose"].landmark:

            features.extend([
                lm.x,
                lm.y,
                lm.z
            ])

    else:
        features.extend([0] * (33 * 3))

    # Left Hand
    if results["left_hand"]:

        for lm in results["left_hand"].landmark:

            features.extend([
                lm.x,
                lm.y,
                lm.z
            ])

    else:
        features.extend([0] * (21 * 3))

    # Right Hand
    if results["right_hand"]:

        for lm in results["right_hand"].landmark:

            features.extend([
                lm.x,
                lm.y,
                lm.z
            ])

    else:
        features.extend([0] * (21 * 3))

    return np.array(
        features,
        dtype=np.float32
    )

In [16]:
#for making the vector shape 75,3 for input

def mediapipe_to_joint_array(results):
    """
    Returns (75, 3) float32 array: 33 pose + 21 left_hand + 21 right_hand joints.
    Each row is [x, y, z]. Missing landmarks are filled with [0, 0, 0].
    """
    joints = []

    if results.get("pose"):
        for lm in results["pose"].landmark:
            joints.append([lm.x, lm.y, lm.z])
    else:
        joints.extend([[0.0, 0.0, 0.0]] * 33)

    if results.get("left_hand"):
        for lm in results["left_hand"].landmark:
            joints.append([lm.x, lm.y, lm.z])
    else:
        joints.extend([[0.0, 0.0, 0.0]] * 21)

    if results.get("right_hand"):
        for lm in results["right_hand"].landmark:
            joints.append([lm.x, lm.y, lm.z])
    else:
        joints.extend([[0.0, 0.0, 0.0]] * 21)

    return np.array(joints, dtype=np.float32)  # (75, 3)


In [ ]:
#Part 1: Graph Class - uilds  (HD-Graph) for the 75-joint MediaPipe skeleton
#Part 2: HDGCN Architecture


# ─── Helper functions ───────────────────────────────────────────────────────

def edge2mat(link, num_node):
    A = np.zeros((num_node, num_node))
    for i, j in link:
        A[j, i] = 1
    return A

def normalize_digraph(A):
    Dl = np.sum(A, axis=0)
    Dn = np.zeros_like(A)
    for i in range(A.shape[1]):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i] ** (-1)
    return np.dot(A, Dn)


# ─── Graph ───────────────────────────────────────────────────────────────────

class Graph:
    """
    Hierarchical graph for 75-joint MediaPipe skeleton.
      Joints  0-32 : Pose (33)
      Joints 33-53 : Left hand (21)
      Joints 54-74 : Right hand (21)
    """
    def __init__(self, num_node=75, center=0, max_hop=4):
        self.num_node = num_node
        self.center   = center
        self.max_hop  = max_hop
        self.edge     = self._build_edges()
        self.groups   = self._get_groups()
        self.edgeset  = self._build_edgeset()
        self.A        = self._build_A()
        print(f"Graph: {len(self.groups)} groups → {len(self.groups)-1} "
              f"hierarchy levels | A.shape = {self.A.shape}")

    def _build_edges(self):
        pose = [
            (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),
            (0,9),(0,10),(9,10),(7,11),(8,12),
            (11,12),(11,13),(13,15),(15,17),(17,19),(19,15),(15,21),
            (12,14),(14,16),(16,18),(18,20),(20,16),(16,22),
            (11,23),(12,24),(23,24),(23,25),(24,26),
            (25,27),(26,28),(27,29),(28,30),(29,31),(30,32),(27,31),(28,32),
        ]
        hand = [
            (0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
            (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
            (0,17),(17,18),(18,19),(19,20),(5,9),(9,13),(13,17),
        ]
        lh    = [(a+33, b+33) for a,b in hand]
        rh    = [(a+54, b+54) for a,b in hand]
        cross = [(15, 33), (16, 54)]            # wrist → hand root
        return pose + lh + rh + cross

    def _bfs(self):
        import collections
        adj = {i: set() for i in range(self.num_node)}
        for i,j in self.edge:
            adj[i].add(j); adj[j].add(i)
        hop = {self.center: 0}
        q   = collections.deque([self.center])
        while q:
            n = q.popleft()
            for nb in adj[n]:
                if nb not in hop:
                    hop[nb] = hop[n] + 1; q.append(nb)
        for i in range(self.num_node):
            if i not in hop: hop[i] = self.max_hop
        return hop

    def _get_groups(self):
        hop    = self._bfs()
        groups = []
        for d in range(self.max_hop + 1):
            if d < self.max_hop:
                nodes = sorted(n for n,h in hop.items() if h == d)
            else:
                nodes = sorted(n for n,h in hop.items() if h >= d)
            if nodes:
                groups.append(nodes)
        return groups

    def _build_edgeset(self):
        G = self.groups
        id_, fwd, rev = [], [], []
        for i in range(len(G) - 1):
            id_.append([(n,n) for n in G[i]+G[i+1]])
            fwd.append([(j,k) for j in G[i] for k in G[i+1]])
            rev.append([(j,k) for j in G[-1-i] for k in G[-2-i]])
        return [[id_[i], fwd[i], rev[-1-i]] for i in range(len(G)-1)]

    def _build_A(self):
        A = []
        for edges in self.edgeset:
            I = edge2mat(edges[0], self.num_node)
            F = normalize_digraph(edge2mat(edges[1], self.num_node))
            R = normalize_digraph(edge2mat(edges[2], self.num_node))
            A.append(np.stack((I, F, R)))
        return np.stack(A)


# ─── Model building blocks ───────────────────────────────────────────────────

def conv_init(conv):
    if conv.weight is not None: nn.init.kaiming_normal_(conv.weight, mode='fan_out')
    if conv.bias   is not None: nn.init.constant_(conv.bias, 0)

def bn_init(bn, scale):
    nn.init.constant_(bn.weight, scale)
    nn.init.constant_(bn.bias, 0)


class TemporalConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, dilation=1):
        super().__init__()
        pad = (kernel_size + (kernel_size-1)*(dilation-1) - 1) // 2
        self.conv = nn.Conv2d(in_ch, out_ch, (kernel_size,1),
                              padding=(pad,0), stride=(stride,1),
                              dilation=(dilation,1), bias=False)
        self.bias = nn.Parameter(torch.zeros(1, out_ch, 1, 1))
        self.bn   = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return self.bn(self.conv(x) + self.bias)


class MultiScale_TemporalConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=5, stride=1,
                 dilations=[1,2], residual=True, residual_kernel_size=1):
        super().__init__()
        assert out_ch % (len(dilations)+2) == 0
        bch = out_ch // (len(dilations)+2)
        ks  = [kernel_size]*len(dilations) if isinstance(kernel_size, int) else kernel_size

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_ch, bch, 1), nn.BatchNorm2d(bch), nn.ReLU(inplace=True),
                TemporalConv(bch, bch, k, stride=stride, dilation=d),
            ) for k,d in zip(ks, dilations)
        ])
        self.branches.append(nn.Sequential(
            nn.Conv2d(in_ch, bch, 1), nn.BatchNorm2d(bch), nn.ReLU(inplace=True),
            nn.MaxPool2d((3,1),(stride,1),(1,0)), nn.BatchNorm2d(bch),
        ))
        self.branches.append(nn.Sequential(
            nn.Conv2d(in_ch, bch, 1, stride=(stride,1)), nn.BatchNorm2d(bch),
        ))
        if   not residual:                       self.residual = lambda x: 0
        elif in_ch==out_ch and stride==1:        self.residual = lambda x: x
        else: self.residual = TemporalConv(in_ch, out_ch, residual_kernel_size, stride=stride)

    def forward(self, x):
        return torch.cat([b(x) for b in self.branches], dim=1) + self.residual(x)


class EdgeConv(nn.Module):
    def __init__(self, in_ch, out_ch, k):
        super().__init__()
        self.k    = k
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch*2, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(inplace=True, negative_slope=0.2),
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):       conv_init(m)
            elif isinstance(m, nn.BatchNorm2d): bn_init(m, 1)

    def _knn(self, x, k):
        inner = -2 * torch.matmul(x.transpose(2,1), x)
        xx    = torch.sum(x**2, dim=1, keepdim=True)
        return (-xx - inner - xx.transpose(2,1)).topk(k=k, dim=-1)[1]

    def _graph_feature(self, x, k):
        N, C, V = x.size()
        idx      = self._knn(x, k)
        base     = torch.arange(N, device=x.device).view(-1,1,1) * V
        idx      = (idx + base).view(-1)
        x_       = rearrange(x, 'n c v -> n v c')
        feat     = rearrange(x_, 'n v c -> (n v) c')[idx].view(N, V, k, C)
        x_       = repeat(x_, 'n v c -> n v k c', k=k)
        return rearrange(torch.cat((feat-x_, x_), dim=3), 'n v k c -> n c v k')

    def forward(self, x, dim=4):
        if dim == 4:
            N, C, T, V = x.size(); x = x.mean(dim=-2)
        x = self.conv(self._graph_feature(x, self.k)).max(dim=-1)[0]
        if dim == 4:
            x = repeat(x, 'n c v -> n c t v', t=T)
        return x


class AHA(nn.Module):
    """Attention-Guided Hierarchy Aggregation."""
    def __init__(self, in_ch, num_layers, groups):
        super().__init__()
        inter            = in_ch // 4
        self.num_layers  = num_layers
        self.layers      = [groups[i]+groups[i+1] for i in range(len(groups)-1)]
        self.conv_down   = nn.Sequential(nn.Conv2d(in_ch, inter, 1), nn.BatchNorm2d(inter), nn.ReLU(inplace=True))
        self.edge_conv   = EdgeConv(inter, inter, k=3)
        self.aggregate   = nn.Conv1d(inter, in_ch, 1)
        self.sigmoid     = nn.Sigmoid()

    def forward(self, x):
        N, C, L, T, V = x.size()
        xt  = self.conv_down(x.max(dim=-2)[0])          # (N,inter,L,V)
        pts = [xt[:,:,i,self.layers[i]].mean(-1,True) for i in range(self.num_layers)]
        pts = torch.cat(pts, dim=2)                      # (N,inter,L)
        att = self.aggregate(self.edge_conv(pts, dim=3)).view(N,C,L,1,1)
        return (x * self.sigmoid(att)).sum(dim=2)        # (N,C,T,V)


class HD_Gconv(nn.Module):
    """Hierarchically Decoupled Graph Convolution."""
    def __init__(self, in_ch, out_ch, A, adaptive=True, residual=True, att=False, groups=None):
        super().__init__()
        self.num_layers = A.shape[0]
        self.num_subset = A.shape[1]
        self.att        = att
        inter           = out_ch // (self.num_subset + 1)

        self.PA         = nn.Parameter(torch.from_numpy(A.astype(np.float32)))
        self.conv_down  = nn.ModuleList()
        self.conv       = nn.ModuleList()

        for _ in range(self.num_layers):
            self.conv_down.append(nn.Sequential(
                nn.Conv2d(in_ch, inter, 1), nn.BatchNorm2d(inter), nn.ReLU(inplace=True)
            ))
            cd = nn.ModuleList()
            for _ in range(self.num_subset):
                cd.append(nn.Sequential(nn.Conv2d(inter, inter, 1), nn.BatchNorm2d(inter)))
            cd.append(EdgeConv(inter, inter, k=5))
            self.conv.append(cd)

        self.aha  = AHA(out_ch, self.num_layers, groups) if att and groups else None
        if   not residual:       self.down = lambda x: 0
        elif in_ch != out_ch:    self.down = nn.Sequential(nn.Conv2d(in_ch,out_ch,1), nn.BatchNorm2d(out_ch))
        else:                    self.down = lambda x: x
        self.bn   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        A, out = self.PA, []
        for i in range(self.num_layers):
            xd = self.conv_down[i](x)
            y  = [self.conv[i][j](torch.einsum('nctu,vu->nctv', xd, A[i,j]))
                  for j in range(self.num_subset)]
            y.append(self.conv[i][-1](xd))
            out.append(torch.cat(y, dim=1))
        out = torch.stack(out, dim=2)                    # (N,out_ch,L,T,V)
        out = self.aha(out) if self.aha else out.sum(2)  # (N,out_ch,T,V)
        return self.relu(self.bn(out) + self.down(x))


class TCN_GCN_unit(nn.Module):
    def __init__(self, in_ch, out_ch, A, stride=1, residual=True,
                 adaptive=True, kernel_size=5, dilations=[1,2], att=True, groups=None):
        super().__init__()
        self.gcn1 = HD_Gconv(in_ch, out_ch, A, adaptive=adaptive, att=att, groups=groups)
        self.tcn1 = MultiScale_TemporalConv(out_ch, out_ch, kernel_size=kernel_size,
                                             stride=stride, dilations=dilations, residual=False)
        self.relu = nn.ReLU(inplace=True)
        if   not residual:               self.residual = lambda x: 0
        elif in_ch==out_ch and stride==1: self.residual = lambda x: x
        else: self.residual = nn.Sequential(nn.Conv2d(in_ch,out_ch,1,stride=(stride,1)), nn.BatchNorm2d(out_ch))

    def forward(self, x):
        return self.relu(self.tcn1(self.gcn1(x)) + self.residual(x))


class HDGCN_Model(nn.Module):
    def __init__(self, num_class, num_point=75, in_channels=3, drop_out=0.5):
        super().__init__()
        g      = Graph(num_node=num_point, center=0, max_hop=4)
        A, grp = g.A, g.groups
        B      = 64

        self.data_bn = nn.BatchNorm1d(in_channels * num_point)
        self.l1  = TCN_GCN_unit(in_channels, B,   A, residual=False, att=False, groups=grp)
        self.l2  = TCN_GCN_unit(B,   B,   A, att=True,  groups=grp)
        self.l3  = TCN_GCN_unit(B,   B,   A, att=True,  groups=grp)
        self.l4  = TCN_GCN_unit(B,   B,   A, att=True,  groups=grp)
        self.l5  = TCN_GCN_unit(B,   B*2, A, stride=2, att=True, groups=grp)
        self.l6  = TCN_GCN_unit(B*2, B*2, A, att=True,  groups=grp)
        self.l7  = TCN_GCN_unit(B*2, B*2, A, att=True,  groups=grp)
        self.l8  = TCN_GCN_unit(B*2, B*4, A, stride=2, att=True, groups=grp)
        self.l9  = TCN_GCN_unit(B*4, B*4, A, att=True,  groups=grp)
        self.l10 = TCN_GCN_unit(B*4, B*4, A, att=True,  groups=grp)
        self.fc  = nn.Linear(B*4, num_class)
        nn.init.normal_(self.fc.weight, 0, math.sqrt(2./num_class))
        bn_init(self.data_bn, 1)
        self.drop_out = nn.Dropout(drop_out)

    def forward(self, x):
        N, C, T, V = x.size()
        x = rearrange(x, 'n c t v -> n (v c) t').contiguous()
        x = self.data_bn(x)
        x = rearrange(x, 'n (v c) t -> n c t v', v=V).contiguous()
        for l in [self.l1,self.l2,self.l3,self.l4,self.l5,
                  self.l6,self.l7,self.l8,self.l9,self.l10]:
            x = l(x)
        x = x.view(N, x.size(1), -1).mean(2)
        return self.fc(self.drop_out(x))

print("HDGCN architecture defined ✓")


In [ ]:
#For Loading Model Weights(HDGCN)



MODEL_PATH = r"c:\Users\Asus\Projects\MetaApp\model-wt\HDGCN.pth"

# Auto-detect number of classes
state_dict  = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
NUM_CLASSES = state_dict['fc.weight'].shape[0]
print(f"Detected {NUM_CLASSES} output classes from checkpoint.")

# Build and load
model = HDGCN_Model(num_class=NUM_CLASSES, num_point=75, in_channels=3)
#model.load_state_dict(state_dict)
result = model.load_state_dict(state_dict, strict=False)
print(f"Unexpected (safely ignored): {result.unexpected_keys}")
print(f"Missing (would be bad!):     {result.missing_keys}")

model.eval().to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded ✓  Parameters: {n_params:,}")

# Load label map  {class_id: class_name}
LABEL_MAP = r"c:\Users\Asus\Projects\MetaApp\model-wt\label_map.json"
if os.path.exists(LABEL_MAP):
    with open(LABEL_MAP) as f:
        raw = json.load(f)
    class_names = {int(v): k for k,v in raw.items()}
    print(f"Label map loaded: {len(class_names)} classes.")
else:
    class_names = {i: f"Sign_{i:03d}" for i in range(NUM_CLASSES)}
    print("WARNING: label_map.json not found in model-wt/.")
    print("         Place it there to see real sign names instead of Sign_000 etc.")


In [20]:
#FOR STGCN

# ─── Graph ───────────────────────────────────────────────────────────────────
class Graph:
    def __init__(self, num_node=75, strategy='spatial'):
        self.num_node = num_node
        self.get_edge()
        self.hop_dis = self.get_hop_distance(self.num_node, self.edge, max_hop=1)
        self.get_adjacency(strategy)

    def get_edge(self):
        upper_body = [
            (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8),
            (9, 10), (11, 12), (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
            (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
            (11, 23), (12, 24), (23, 24)
        ]
        left_hand = [
            (33, 34), (34, 35), (35, 36), (36, 37),
            (33, 38), (38, 39), (39, 40), (40, 41),
            (33, 42), (42, 43), (43, 44), (44, 45),
            (33, 46), (46, 47), (47, 48), (48, 49),
            (33, 50), (50, 51), (51, 52), (52, 53)
        ]
        right_hand = [
            (54, 55), (55, 56), (56, 57), (57, 58),
            (54, 59), (59, 60), (60, 61), (61, 62),
            (54, 63), (63, 64), (64, 65), (65, 66),
            (54, 67), (67, 68), (68, 69), (69, 70),
            (54, 71), (71, 72), (72, 73), (73, 74)
        ]
        body_hand = [(15, 33), (16, 54)]
        self.edge = upper_body + left_hand + right_hand + body_hand
        self.self_link = [(i, i) for i in range(self.num_node)]
        self.edge += self.self_link
        self.center = 0

    def get_hop_distance(self, num_node, edge, max_hop=1):
        A = np.zeros((num_node, num_node))
        for i, j in edge:
            if i < num_node and j < num_node:
                A[j, i] = 1
                A[i, j] = 1
        hop_dis = np.zeros((num_node, num_node)) + np.inf
        transfer_mat = [np.linalg.matrix_power(A, d) for d in range(max_hop + 1)]
        arrive_mat = (np.stack(transfer_mat) > 0)
        for d in range(max_hop, -1, -1):
            hop_dis[arrive_mat[d]] = d
        return hop_dis

    def get_adjacency(self, strategy):
        valid_hop = self.hop_dis < np.inf
        if strategy == 'spatial':
            node_center = self.hop_dis[:, self.center]
            A = np.zeros((3, self.num_node, self.num_node))
            for i in range(self.num_node):
                for j in range(self.num_node):
                    if self.hop_dis[j, i] == 1:
                        if node_center[j] == node_center[i]:
                            A[0, j, i] = 1
                        elif node_center[j] > node_center[i]:
                            A[1, j, i] = 1
                        else:
                            A[2, j, i] = 1
                    elif self.hop_dis[j, i] == 0:
                        A[0, j, i] = 1
            for i in range(3):
                D = A[i].sum(axis=1)
                D[D == 0] = 1
                A[i] = A[i] / D[:, None]
            self.A = A

# ─── Model building blocks ───────────────────────────────────────────────────
class ConvTemporalGraphical(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, t_kernel_size=1, t_stride=1, t_padding=0, t_dilation=1, bias=True):
        super().__init__()
        self.kernel_size = kernel_size
        self.conv = nn.Conv2d(in_channels, out_channels * kernel_size, kernel_size=(t_kernel_size, 1), padding=(t_padding, 0), stride=(t_stride, 1), dilation=(t_dilation, 1), bias=bias)

    def forward(self, x, A):
        assert A.size(0) == self.kernel_size
        x = self.conv(x)
        n, kc, t, v = x.size()
        x = x.view(n, self.kernel_size, kc // self.kernel_size, t, v)
        x = torch.einsum('nkctv,kvw->nctw', x, A)
        return x.contiguous()

class st_gcn_block(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dropout=0.5, residual=True):
        super().__init__()
        assert len(kernel_size) == 2
        assert kernel_size[0] % 2 == 1
        padding = ((kernel_size[0] - 1) // 2, 0)

        self.gcn = ConvTemporalGraphical(in_channels, out_channels, kernel_size[1])
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, (kernel_size[0], 1), (stride, 1), padding),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout, inplace=True)
        )
        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, A):
        res = self.residual(x)
        x = self.gcn(x, A)
        x = self.tcn(x) + res
        return self.relu(x)

class ST_GCN_Model(nn.Module):
    def __init__(self, in_channels, num_class, graph_args, edge_importance_weighting=True, **kwargs):
        super().__init__()
        self.graph = Graph(**graph_args)
        A = torch.tensor(self.graph.A, dtype=torch.float32, requires_grad=False)
        self.register_buffer('A', A)

        spatial_kernel_size = A.size(0)
        temporal_kernel_size = 9
        kernel_size = (temporal_kernel_size, spatial_kernel_size)

        self.data_bn = nn.BatchNorm1d(in_channels * A.size(1))

        self.st_gcn_networks = nn.ModuleList((
            st_gcn_block(in_channels, 64, kernel_size, 1, residual=False),
            st_gcn_block(64, 64, kernel_size, 1),
            st_gcn_block(64, 64, kernel_size, 1),
            st_gcn_block(64, 64, kernel_size, 1),
            st_gcn_block(64, 128, kernel_size, 2),
            st_gcn_block(128, 128, kernel_size, 1),
            st_gcn_block(128, 128, kernel_size, 1),
            st_gcn_block(128, 256, kernel_size, 2),
            st_gcn_block(256, 256, kernel_size, 1),
            st_gcn_block(256, 256, kernel_size, 1),
        ))

        if edge_importance_weighting:
            self.edge_importance = nn.ParameterList([
                nn.Parameter(torch.ones(self.A.size()))
                for i in self.st_gcn_networks
            ])
        else:
            self.edge_importance = [1] * len(self.st_gcn_networks)

        self.fcn = nn.Conv2d(256, num_class, kernel_size=1)

    def forward(self, x):
        # 4D Tensor Input: (N, C, T, V)
        N, C, T, V = x.size()
        
        # BN expects (N, V*C, T)
        x = x.permute(0, 3, 1, 2).contiguous() # (N, V, C, T)
        x = x.view(N, V * C, T)
        x = self.data_bn(x)
        x = x.view(N, V, C, T).permute(0, 2, 3, 1).contiguous() # (N, C, T, V)

        # ST-GCN Branches
        for gcn, importance in zip(self.st_gcn_networks, self.edge_importance):
            x = gcn(x, self.A * importance)

        # Global pooling
        x = F.avg_pool2d(x, x.size()[2:]) # (N, 256, 1, 1)
        x = self.fcn(x) # (N, NUM_CLASSES, 1, 1)
        x = x.view(x.size(0), -1) # (N, NUM_CLASSES)
        
        return x

print("STGCN architecture defined ✓")

# ─── Load Model Weights ──────────────────────────────────────────────────────
MODEL_PATH = r"c:\Users\Asus\Projects\MetaApp\model-wt\ST-GCN.pth"
device = torch.device('cpu')

# Load dictionary to find number of classes
state_dict = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)

# Dynamically find the classification layer name to determine NUM_CLASSES
fcn_weight_key = [k for k in state_dict.keys() if 'fcn.weight' in k][0]
NUM_CLASSES = state_dict[fcn_weight_key].shape[0]
print(f"Detected {NUM_CLASSES} output classes from checkpoint.")

# Build model
model = ST_GCN_Model(in_channels=3, num_class=NUM_CLASSES, graph_args={'num_node': 75, 'strategy': 'spatial'})

# Load weights (using strict=False to bypass any harmless duplicate keys)
result = model.load_state_dict(state_dict, strict=False)
print(f"Unexpected keys (safely ignored): {result.unexpected_keys}")
print(f"Missing keys: {result.missing_keys}")
model.eval().to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded ✓  Parameters: {n_params:,}")

# Load label map {class_id: class_name}
LABEL_MAP = r"c:\Users\Asus\Projects\MetaApp\model-wt\label_map.json"
if os.path.exists(LABEL_MAP):
    with open(LABEL_MAP) as f:
        raw = json.load(f)
    class_names = {int(v): k for k,v in raw.items()}
    print(f"Label map loaded: {len(class_names)} classes.")
else:
    class_names = {i: f"Sign_{i:03d}" for i in range(NUM_CLASSES)}
    print("WARNING: label_map.json not found in model-wt/.")


STGCN architecture defined ✓
Detected 120 output classes from checkpoint.
Unexpected keys (safely ignored): []
Missing keys: []
Model loaded ✓  Parameters: 3,264,552
Label map loaded: 120 classes.


In [21]:
mp_draw = mp.solutions.drawing_utils


def draw_mediapipe(frame, results):

    if results["pose"]:

        mp_draw.draw_landmarks(
            frame,
            results["pose"],
            mp_holistic.POSE_CONNECTIONS
        )

    if results["left_hand"]:

        mp_draw.draw_landmarks(
            frame,
            results["left_hand"],
            mp.solutions.hands.HAND_CONNECTIONS
        )

    if results["right_hand"]:

        mp_draw.draw_landmarks(
            frame,
            results["right_hand"],
            mp.solutions.hands.HAND_CONNECTIONS
        )

    return frame

In [ ]:
#run before connection of glasses
#Run the app in the phone with phone connectd iwth laptop and on the same network

# ── Meta Glasses WebSocket Server ──────────────────────────────────────────
# Starts a WebSocket server on port 8765 in a background thread.
# The Meta glasses Android app connects TO this server and pushes JPEG frames.
# Each WebSocket message is one complete JPEG — no byte scanning needed.

META_HOST = "0.0.0.0"
META_PORT = 8765

meta_frame_queue = queue.Queue(maxsize=2)   # small buffer; drop old frames if full
meta_stop_event  = threading.Event()
meta_server_thread = None


async def _meta_handler(websocket):
    print("Meta glasses connected!")
    try:
        async for message in websocket:
            jpg = np.frombuffer(message, dtype=np.uint8)
            frame = cv2.imdecode(jpg, cv2.IMREAD_COLOR)
            if frame is None:
                continue
            # Drop oldest frame if queue is full to avoid lag
            if meta_frame_queue.full():
                try:
                    meta_frame_queue.get_nowait()
                except queue.Empty:
                    pass
            meta_frame_queue.put(frame)
    except websockets.exceptions.ConnectionClosed:
        print("Meta glasses disconnected.")


async def _meta_server_coroutine():
    async with websockets.serve(_meta_handler, META_HOST, META_PORT):
        print(f"WebSocket server listening on ws://{META_HOST}:{META_PORT}")
        print("Waiting for Meta glasses to connect...")
        # Keep running until stop event is set
        while not meta_stop_event.is_set():
            await asyncio.sleep(0.1)


def _meta_server_thread_fn():
    asyncio.run(_meta_server_coroutine())


# Start server in a daemon background thread
meta_stop_event.clear()
meta_server_thread = threading.Thread(target=_meta_server_thread_fn, daemon=True)
meta_server_thread.start()


In [ ]:
#TO SHUT DOWN WEB SOCKET!
# Run this cell to shut down the WebSocket server when done
meta_stop_event.set()
print("Meta WebSocket server stopped.")


In [ ]:
# The Pipeline 
# Utilizes Sliding Window

# Meta Frame -> Mediapipe -> Joint_array ->  frame_buffer -> stack → (80,75,3) ->
# transpose → (3,80,75) -> unsqueeze → (1,3,80,75) -> HDGCN model ->  softmax → class + confidence -> overlay on video frame → display
                                               
import io, queue
from IPython.display import Image as IPImage

# ─── Parameters ─────────────────────────────
NUM_FRAMES = 80   # must stay 80 (training sequence length)
STRIDE     = 8    # inference every N frames — tune if needed

# ─── Buffers ────────────────────────────────
frame_buffer           = deque(maxlen=NUM_FRAMES)
frames_since_last_pred = 0
current_prediction     = f"Collecting frames…"
current_confidence     = 0.0
frame_count            = 0

display_handle = display(None, display_id=True)
print(f"Pipeline starting  |  First prediction after {NUM_FRAMES} frames, then every {STRIDE} frames.")
print("Press the ■ Stop button to stop.\n")

try:
    while True:
        # 1. Grab frame from Meta glasses WebSocket queue
        try:
            frame = meta_frame_queue.get(timeout=5)
        except queue.Empty:
            print("No frame received in 5s — is the Meta glasses app connected?")
            continue
        if frame is None:
            continue
        frame_count += 1

        # 2. MediaPipe feature extraction
        frame_rgb  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results    = extract_pose(frame_rgb)

        # 3. Build (75, 3) joint array and add to sliding window
        joint_arr  = mediapipe_to_joint_array(results)   # (75, 3)
        frame_buffer.append(joint_arr)
        frames_since_last_pred += 1

        # 4. Run inference (buffer full + stride met)
        if len(frame_buffer) == NUM_FRAMES and frames_since_last_pred >= STRIDE:
            frames_since_last_pred = 0

            seq    = np.stack(list(frame_buffer), axis=0)               # (80, 75, 3)
            seq    = seq.transpose(2, 0, 1)                              # (3, 80, 75) = (C, T, V)
            tensor = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)  # (1,3,80,75)

            with torch.no_grad():
                logits = model(tensor)                                   # (1, num_classes)
                probs  = torch.softmax(logits, dim=1)
                conf, idx = probs.max(dim=1)
                current_confidence = conf.item()
                current_prediction = class_names[idx.item()]

        # 5. Draw MediaPipe skeleton on frame
        vis = cv2.cvtColor(draw_mediapipe(frame.copy(), results), cv2.COLOR_BGR2RGB)

        # 6. Overlay prediction text
        label = (f"{current_prediction}  ({current_confidence*100:.1f}%)"
                 if len(frame_buffer) == NUM_FRAMES
                 else f"Collecting:  {len(frame_buffer)} / {NUM_FRAMES}")
        cv2.putText(vis, label, (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

        # 7. Display in notebook (flicker-free)
        buf = io.BytesIO()
        Image.fromarray(vis).save(buf, format='JPEG', quality=80)
        buf.seek(0)
        display_handle.update(IPImage(data=buf.getvalue()))

        print(f"\rFrame {frame_count:05d} | Buffer {len(frame_buffer):3d}/{NUM_FRAMES} | "
              f"{current_prediction} ({current_confidence*100:.1f}%)", end="", flush=True)

except KeyboardInterrupt:
    print("\n\nPipeline stopped.")


In [ ]:
#OLD
#FOR META GLASSES:Preview of BOX AND  mediapipe only

frame_count = 0

# CHANGED: reads decoded frames from Meta glasses WebSocket queue 
while True:

    try:
        frame = meta_frame_queue.get(timeout=5)
    except queue.Empty:
        print("No frame received in 5 s — is the glasses app connected?")
        continue

    # Everything below this line is IDENTICAL to the DroidCam version 

    if frame is None:
        continue

    frame_rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    results = extract_pose(frame_rgb)

    feature_vector = mediapipe_to_vector(results)

    frame = draw_mediapipe(
        frame,
        results
    )

    frame_rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    frame_count += 1

    clear_output(wait=True)

    print("Frame:", frame_count)
    print("Feature Length:", len(feature_vector))

    display(
        Image.fromarray(frame_rgb)
    )


In [ ]:
#OLD
#FOR DROID CAM
frame_count = 0

for chunk in stream.iter_content(chunk_size=4096):

    buffer += chunk

    start = buffer.find(b'\xff\xd8')
    end = buffer.find(b'\xff\xd9')

    if start != -1 and end != -1:

        jpg = buffer[start:end+2]
        buffer = buffer[end+2:]

        frame = cv2.imdecode(
            np.frombuffer(jpg, np.uint8),
            cv2.IMREAD_COLOR
        )

        if frame is None:
            continue

        frame_rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        results = extract_pose(frame_rgb)

        feature_vector = mediapipe_to_vector(results)

        frame = draw_mediapipe(
            frame,
            results
        )

        frame_rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frame_count += 1

        clear_output(wait=True)

        print("Frame:", frame_count)
        print("Feature Length:", len(feature_vector))

        display(
            Image.fromarray(frame_rgb)
        )

use pretrained classififers which is finetuned with our own dataset show all the classifier options and allow user to select one

In [ ]:
import torch

ckpt = torch.load(r"F:\iiit sricity\ISL_Implementation\Meta_glass\models\CTR-GCN.pth",map_location="cpu")

print(type(ckpt))

if isinstance(ckpt, dict):
    print(ckpt.keys())

In [ ]:
import torch

ckpt = torch.load(r"F:\iiit sricity\ISL_Implementation\Meta_glass\models\CTR-GCN.pth",map_location="cpu")

print("fc.weight shape =", ckpt["fc.weight"].shape)
print("fc.bias shape   =", ckpt["fc.bias"].shape)